# From a sampled file to geometry

Write a small synthetic XDMF/HDF5 pair, verify axis order and units, compare
interpolation against a known nonlinear field, and trace within its bounds.
All generated data lives in a temporary directory. No simulation path is assumed.

Run top to bottom in a fresh Python kernel after installing `.[examples]` from
    the repository root. No external data, network access, Qt or PyVista is required.
    Stored outputs are cleared; figures and assertions are generated by the cells.
    [Notebook index](README.md) · [Analysis guide](../../docs/geometry_analysis.md)

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import h5py
import matplotlib.pyplot as plt
from mageometry import load_xdmf, trace_field_lines
from mageometry.geometry import field_line_curvature

def reference(x, y, z):
    x, y, z = np.broadcast_arrays(x, y, z)
    return -y, x, 1. + .2*x*x

axes = (np.linspace(.5, 2., 17), np.linspace(-1., 1., 19), np.linspace(-1., 1., 21))
coords = np.meshgrid(*axes, indexing='ij')
components = reference(*coords)
workspace = tempfile.TemporaryDirectory(prefix='mageometry-notebook-')
directory = Path(workspace.name)

## Write the documented reader convention

In memory, arrays have shape (nx, ny, nz). This XDMF uniform-grid example stores
HDF5 arrays as (nz, ny, nx), and origin/spacing in z/y/x order. Unequal dimensions
make an accidental transpose detectable. This is generated test data, not a
claim about every simulation's convention; consult the
[format guide](../../docs/simulation_data_formats.md).

In [ ]:
nx, ny, nz = (len(axis) for axis in axes)
with h5py.File(directory / 'field.h5', 'w') as output:
    for name, values in zip(('BX', 'BY', 'BZ'), components):
        output[name] = values.transpose(2, 1, 0)
origin = ' '.join(str(axis[0]) for axis in axes[::-1])
spacing = ' '.join(str(axis[1] - axis[0]) for axis in axes[::-1])
attributes = ''.join(
    f'<Attribute Name="{name}" AttributeType="Scalar" Center="Node">'
    f'<DataItem Dimensions="{nz} {ny} {nx}" NumberType="Float" Precision="8" '
    f'Format="HDF">field.h5:/{name}</DataItem></Attribute>'
    for name in ('BX', 'BY', 'BZ'))
xml = (f'<Xdmf Version="2.0"><Domain><Grid Name="Synthetic" GridType="Uniform">'
       f'<Topology TopologyType="3DCORECTMesh" NumberOfElements="{nz} {ny} {nx}"/>'
       f'<Geometry GeometryType="ORIGIN_DXDYDZ">'
       f'<DataItem Name="Origin" Dimensions="3" Format="XML">{origin}</DataItem>'
       f'<DataItem Name="Spacing" Dimensions="3" Format="XML">{spacing}</DataItem>'
       f'</Geometry>{attributes}</Grid></Domain></Xdmf>')
(directory / 'field.xmf').write_text(xml)
grid = load_xdmf(directory / 'field.xmf')
np.testing.assert_allclose(grid.b, np.stack(components, axis=-1))
for actual, expected in zip((grid.x, grid.y, grid.z), axes):
    np.testing.assert_allclose(actual, expected, atol=1e-12)
grid.metadata.update(coordinate_system='Cartesian', length_unit='m', field_unit='T')
print(grid.shape, grid.bounds)

## Interpolation and derivatives are separate error sources

Metadata labels do not rescale arrays; this example chooses m and T as its native
units. Cubic interpolation is available with at least four nodes per axis.
Compare it with linear interpolation on these smooth data; it is not automatically
better for discontinuities or noisy inputs. A smaller derivative step cannot
recover structure missing from the grid.

In [ ]:
rng = np.random.default_rng(42)
points = rng.uniform([.7, -.7, -.7], [1.8, .7, .7], size=(50, 3))
expected = np.array(reference(*points.T))
fields = {method: grid.field(method=method) for method in ('linear', 'cubic')}
for method, field in fields.items():
    field_error = np.max(np.abs(np.array(field(*points.T)) - expected))
    print(method, 'max magnetic error [T]:', field_error)
    assert field_error < .002
steps = (.2, .1, .05)
fig, ax = plt.subplots()
truth = field_line_curvature(reference, *points.T, delta=.001)
for method, field in fields.items():
    errors = [np.nanmax(np.abs(field_line_curvature(field, *points.T, delta=h) - truth))
              for h in steps]
    ax.plot(steps, errors, 'o-', label=method)
ax.set(xlabel='Difference step [m]', ylabel='Maximum curvature error [1/m]')
ax.legend()
plt.show()

## Boundaries and tracing

The default interpolant returns NaN outside the grid. Derivative stencils need
room inside the data domain too. Retain NaN masks rather than filling them with
zero. Pass the grid bounds to tracing to distinguish a boundary stop from an
unexpected invalid sample.

In [ ]:
field = fields['linear']
assert np.all(np.isnan(field(3., 0., 0.)))
assert np.isnan(field_line_curvature(field, 3., 0., 0., delta=.1))
trace = trace_field_lines(field, 1., 0., 0., direction='both', ds=.02,
                         bounds=grid.bounds, max_steps=300)
print('Forward/backward status:', trace.status, trace.status_backward)
assert trace.status[0] == 1 and trace.status_backward[0] == 1
workspace.cleanup()

## Bring your own snapshot

Replace file generation with `load_xdmf`, `load_hdf5`, `load_vtk`, or construct a
`GriddedField` directly for other readers. Declare coordinates and units only
after confirming their meaning. Reader stride and viewer preview coarsening alter
numerical sampling; check both. Do not assume a dipole background for an MHD
snapshot without establishing that convention.

[Notebook 06](06_visualization.ipynb) shows reusable plotting, and
[notebook 8](08_transverse_geometry_and_fac.ipynb) explains which diagnostics
need a valid curvature normal.